# 02. Register and Login (Sec. IV-A to IV-C, Fig. 5)

Three servers $S_1, S_2, S_3$ share a TOPRF key with threshold $t = 2$. The user "alice" registers
with a password. At Login she contacts two servers and re-derives the same $K_u$ and the same
Dilithium key pair from the password alone. Each step prints what crossed the wire, with sizes in
bytes, and what each party holds afterwards.

In [1]:
import statistics
import time

from ris_sem_project import protocol
from ris_sem_project.display import bits, payload_size, peek, table
from ris_sem_project.params import PARAM_I, setup
from ris_sem_project.parties import ProtocolAbort
from ris_sem_project.parties.qpase_server import QPASEServer
from ris_sem_project.parties.qpase_user import QPASEUser
from ris_sem_project.primitives.ring import Ring
from ris_sem_project.primitives.secret_sharing import NonIntegerLagrangeError, lagrange_at_zero

PASSWORD = "correct horse battery staple"


def attempts(user_id="alice"):
    return {f"S_{i}": server.users[user_id].fail_count for i, server in servers.items()}

## Setup (Sec. IV-A)

$pp \leftarrow Setup(1^\kappa)$ samples the public $a, a_0, a_1 \in R_q^{1 \times \ell}$ and fixes
$N$, $t$ and $\mu$, the limit on Login attempts per user and epoch. Login step L1 adds 1 to
$\mu_u$ at every contacted server on every attempt, so $\mu$ here leaves room for this notebook's
logins plus a few wrong guesses.

In [2]:
pp = setup(PARAM_I, N=3, t=2, mu=14)
ring = Ring(pp.q, pp.n)
print(f"q = {pp.q}, n = {pp.n}, ell = {pp.ell}, p = {pp.p}, sigma = {pp.sigma}, L = {pp.L}")
print(f"N = {pp.N}, t = {pp.t}, mu = {pp.mu}")
print(f"a = {peek(pp.a)}, a_0 = {peek(pp.a0)}, a_1 = {peek(pp.a1)}")

q = 268435399, n = 512, ell = 28, p = 2, sigma = 3.2, L = 128
N = 3, t = 2, mu = 14
a = 7e10…e90a (57344 B), a_0 = 9760…2000 (57344 B), a_1 = e9df…e60f (57344 B)


## DKG among the servers (Definition 12)

Each $S_i$ runs Genshare and sends $[s_i]_j$ to every $S_j$. Each $S_j$ runs Genkey,
$k_j = \sum_i [s_i]_j$, and publishes $pk_j = a k_j + 2e_j$. Only $S_j$ holds $k_j$.

In [3]:
servers = {i: QPASEServer(pp, i) for i in range(1, pp.N + 1)}
dkg = protocol.dkg(servers)
print(table(
    ("message", "sender", "receiver", "[s_i]_j"),
    [("DKGShare", f"S_{i}", f"S_{j}", peek(share.share)) for i, row in dkg.shares.items() for j, share in row.items()],
))
print()
print(table(("server", "pk_j = a k_j + 2e_j", "k_j (never sent)"), [
    (f"S_{j}", peek(ring.to_bytes(server.key.pk_i)), peek(ring.to_bytes(server.key.k_i)))
    for j, server in servers.items()
]))

message   sender  receiver  [s_i]_j
--------  ------  --------  ------------------
DKGShare  S_1     S_1       892d…5b0e (2048 B)
DKGShare  S_1     S_2       4a5b…b70c (2048 B)
DKGShare  S_1     S_3       0b89…130b (2048 B)
DKGShare  S_2     S_1       1f5a…be01 (2048 B)
DKGShare  S_2     S_2       3ab4…7c03 (2048 B)
DKGShare  S_2     S_3       550e…3b05 (2048 B)
DKGShare  S_3     S_1       4758…1405 (2048 B)
DKGShare  S_3     S_2       8eb0…280a (2048 B)
DKGShare  S_3     S_3       d508…3c0f (2048 B)

server  pk_j = a k_j + 2e_j  k_j (never sent)
------  -------------------  ------------------
S_1     2a12…ff08 (57344 B)  28e0…2e05 (2048 B)
S_2     d8a2…020e (57344 B)  4bc0…5c0a (2048 B)
S_3     8e33…0603 (57344 B)  6ea0…8a0f (2048 B)


## Register step 1: TOPRF with every server (Sec. IV-B)

Each $S_i$ first checks that $ID_u$ is new. U sends $b = a r + a_x + 2e$ to all $N$ servers, where
$a_x$ encodes the password. $S_i$ answers $b_{k_i} = b k_i + 2e'_i$ with its $pk_i$. U combines $t$ or
more answers into $F = a_x K + 2e''$, computes the hint $\sigma = S(F)$ and extracts $K_u = E(F, \sigma)$.

In [4]:
alice = QPASEUser(pp, "alice", PASSWORD)
registered = protocol.register(alice, servers)
print(table(("server", "U -> S_i: b", "S_i -> U: b_k_i", "S_i -> U: pk_i", "response total"), [
    (
        f"S_{i}",
        peek(registered.requests[i].blinded_input),
        peek(response.evaluation),
        peek(response.server_pk),
        f"{payload_size(response)} B",
    )
    for i, response in registered.responses.items()
]))
print(f"TOPRF request per server: {payload_size(registered.requests[1])} B (user id and b)")
print(f"K_u = E(F, sigma): {peek(registered.k_u)}, {bits(registered.k_u)}")

server  U -> S_i: b          S_i -> U: b_k_i      S_i -> U: pk_i       response total
------  -------------------  -------------------  -------------------  --------------
S_1     7170…4b0e (57344 B)  c4de…2308 (57344 B)  2a12…ff08 (57344 B)  114688 B
S_2     7170…4b0e (57344 B)  963c…fc00 (57344 B)  d8a2…020e (57344 B)  114688 B
S_3     7170…4b0e (57344 B)  1b9a…d409 (57344 B)  8e33…0603 (57344 B)  114688 B
TOPRF request per server: 57349 B (user id and b)
K_u = E(F, sigma): 2a6e…3c5a (1792 B), 14336 bits, 7214 ones: 0010101001101110…0011110001011010


## Register steps 2-4

U derives $(pk_u, sk_u) \leftarrow Gen(1^\kappa, K_u)$ (ML-DSA-44 seeded from $HKDF(K_u, \text{sig})$)
and sends $pk_u$ to every server with the hint $\sigma$ (plan decision 4: the paper's Fig. 2 has no
stored hint). $S_i$ stores $(ID_u, pk_u, \sigma, \mu_u = 0)$. U keeps only $ID_u$ and $psw_u$.

In [5]:
key_message = registered.key_message
print(f"U -> every S_i: pk_u = {peek(key_message.user_pk)}, sigma = {peek(key_message.hint)}")
print(f"RegisterKeyMessage: {payload_size(key_message)} B")
print()
print(table(("server", "ID_u", "pk_u", "sigma", "mu_u"), [
    (f"S_{i}", record.user_id, peek(record.user_pk), peek(record.hint), record.fail_count)
    for i, server in servers.items()
    for record in [server.users["alice"]]
]))

U -> every S_i: pk_u = 1ea4…f5f2 (1312 B), sigma = f01c…3558 (1792 B)
RegisterKeyMessage: 3109 B

server  ID_u   pk_u                sigma               mu_u
------  -----  ------------------  ------------------  ----
S_1     alice  1ea4…f5f2 (1312 B)  f01c…3558 (1792 B)  0
S_2     alice  1ea4…f5f2 (1312 B)  f01c…3558 (1792 B)  0
S_3     alice  1ea4…f5f2 (1312 B)  f01c…3558 (1792 B)  0


## Login with $\{S_1, S_2\}$ (Sec. IV-C)

L1: each contacted $S_i$ checks $\mu_u < \mu$, sets $\mu_u \leftarrow \mu_u + 1$, evaluates the TOPRF
and returns $pk_u$ and $\sigma$. L2: U extracts $K_u = E(F, \sigma)$, derives $(pk'_u, sk'_u)$, and
re-asks for the password unless $pk'_u = pk_u$. Otherwise U signs $sig_u = Sign(sk'_u, (ID_u, sid))$
for each server's fresh $sid$. L3: $S_i$ verifies $sig_u$.

In [6]:
alice = QPASEUser(pp, "alice", PASSWORD)
login_12 = protocol.login(alice, servers, (1, 2))
print(table(("server", "S_i -> U: b_k_i", "pk_u", "sigma", "L2: sid", "U -> S_i: sig_u"), [
    (
        f"S_{i}",
        peek(response.evaluation),
        peek(response.user_pk),
        peek(response.hint),
        peek(login_12.proofs[i].session_id),
        peek(login_12.proofs[i].signature),
    )
    for i, response in login_12.responses.items()
]))
print(f"L1 response per server: {payload_size(login_12.responses[1])} B, L2 proof per server: {payload_size(login_12.proofs[1])} B")
print(f"K_u at Login {{1, 2}} == K_u at Register: {login_12.k_u == registered.k_u}")
print(f"L3 accepted by S_1 and S_2, mu_u: {attempts()}")

server  S_i -> U: b_k_i      pk_u                sigma               L2: sid           U -> S_i: sig_u
------  -------------------  ------------------  ------------------  ----------------  ------------------
S_1     5fd1…9507 (57344 B)  1ea4…f5f2 (1312 B)  f01c…3558 (1792 B)  417b…886c (32 B)  def2…2b41 (2420 B)
S_2     5341…a203 (57344 B)  1ea4…f5f2 (1312 B)  f01c…3558 (1792 B)  3287…1716 (32 B)  49eb…3348 (2420 B)
L1 response per server: 117792 B, L2 proof per server: 2457 B
K_u at Login {1, 2} == K_u at Register: True
L3 accepted by S_1 and S_2, mu_u: {'S_1': 1, 'S_2': 1, 'S_3': 0}


## Login with $\{S_2, S_3\}$

A different pair of servers holds different shares, and the TOPRF noise is fresh, yet $E(F, \sigma)$
gives the same $K_u$ and therefore the same $pk_u$.

In [7]:
login_23 = protocol.login(QPASEUser(pp, "alice", PASSWORD), servers, (2, 3))
print(f"K_u {{2, 3}}: {peek(login_23.k_u)}")
print(f"K_u {{1, 2}}: {peek(login_12.k_u)}")
print(f"identical K_u: {login_23.k_u == login_12.k_u}, identical pk_u: {login_23.user_pk == login_12.user_pk}")
print(f"mu_u: {attempts()}")

K_u {2, 3}: 2a6e…3c5a (1792 B)
K_u {1, 2}: 2a6e…3c5a (1792 B)
identical K_u: True, identical pk_u: True
mu_u: {'S_1': 1, 'S_2': 2, 'S_3': 1}


## $\{S_1, S_3\}$ is refused

For $\{1, 3\}$ the Lagrange coefficients at 0 are $3/2$ and $-1/2$. The combined noise would no longer
be a small even integer, so U refuses this subset before contacting any server (plan decision 3).

In [8]:
for subset in ((1, 2), (2, 3), (1, 3)):
    try:
        print(f"{subset}: lambda = {lagrange_at_zero(subset)}")
    except NonIntegerLagrangeError as error:
        print(f"{subset}: {error}")
before = attempts()
try:
    protocol.login(QPASEUser(pp, "alice", PASSWORD), servers, (1, 3))
    refused_13 = False
except ValueError as error:
    refused_13 = True
    print(f"login with (1, 3) refused by U: {type(error).__name__}: {error}")
print(f"mu_u unchanged: {attempts() == before} {attempts()}")

(1, 2): lambda = {1: 2, 2: -1}
(2, 3): lambda = {2: 3, 3: -2}
(1, 3): lambda_1 = 3/2 for ids [1, 3]
login with (1, 3) refused by U: NonIntegerLagrangeError: lambda_1 = 3/2 for ids [1, 3]
mu_u unchanged: True {'S_1': 1, 'S_2': 2, 'S_3': 1}


## Wrong password

Someone who knows $ID_u$ but not the password runs L1 with $\{S_1, S_2\}$. The servers cannot tell:
they count the attempt and answer. U's $K'_u$ differs from $K_u$, so $pk'_u \neq pk_u$ and L2 stops
before any signature is sent. The steps are run one by one to show where it stops.

In [9]:
guesser = QPASEUser(pp, "alice", "correct horse battery stable")
before = attempts()
requests = guesser.toprf_request((1, 2))
responses = {i: servers[i].toprf_evaluate(request) for i, request in requests.items()}
k_guess = guesser.toprf_finalize(list(responses.values()))
differing = sum(bin(a ^ b).count("1") for a, b in zip(k_guess, registered.k_u))
print(f"K'_u from the wrong password: {peek(k_guess)}, {differing} of {8 * len(k_guess)} bits differ from K_u")
try:
    guesser.login_verify(k_guess, responses[1].user_pk, servers[1].new_session("alice"))
    wrong_rejected = False
except ProtocolAbort as error:
    wrong_rejected = True
    print(f"L2 at U: {error}")
print(f"mu_u before: {before}")
print(f"mu_u after:  {attempts()}")
wrong_counted = attempts()["S_1"] == before["S_1"] + 1 and attempts()["S_2"] == before["S_2"] + 1

K'_u from the wrong password: 04e4…f486 (1792 B), 7172 of 14336 bits differ from K_u


L2 at U: pk_u' != pk_u: wrong password, re-enter it and run L1 again
mu_u before: {'S_1': 1, 'S_2': 2, 'S_3': 1}
mu_u after:  {'S_1': 2, 'S_2': 3, 'S_3': 1}


## The same password always gives the same $K_u$

Each Login draws fresh $r$, $e$ and fresh server noise $e'_i$. The cell repeats the correct Login
$K = 10$ times, alternating $\{1, 2\}$ and $\{2, 3\}$, and compares every $K_u$ and $pk_u$ byte for byte
with Register's.

In [10]:
K = 10
repeat_keys, repeat_pks, login_times = [], [], []
for attempt in range(K):
    subset = (1, 2) if attempt % 2 == 0 else (2, 3)
    start = time.perf_counter()
    run = protocol.login(QPASEUser(pp, "alice", PASSWORD), servers, subset)
    login_times.append(time.perf_counter() - start)
    repeat_keys.append(run.k_u)
    repeat_pks.append(run.user_pk)
same_k_u = sum(k == registered.k_u for k in repeat_keys)
same_pk = sum(pk == key_message.user_pk for pk in repeat_pks)
print(f"{same_k_u}/{K} logins gave byte-identical K_u")
print(f"{same_pk}/{K} logins gave byte-identical pk_u")
print(f"Login (L1 to L3, t = 2 servers) over {K} runs: median {statistics.median(login_times):.2f} s")
print(f"mu_u: {attempts()}")

10/10 logins gave byte-identical K_u
10/10 logins gave byte-identical pk_u
Login (L1 to L3, t = 2 servers) over 10 runs: median 2.55 s
mu_u: {'S_1': 7, 'S_2': 13, 'S_3': 6}


## The limit $\mu$ stops online guessing

The guesser keeps trying passwords from a list against $\{S_1, S_2\}$. Each try costs one L1 at each
contacted server. Once $\mu_u = \mu$ on a server, that server aborts L1, and every subset with integer
Lagrange coefficients contains $S_2$. After that even the correct password is refused until a key
update resets $\mu_u$ (Sec. IV-F).

In [11]:
def guess(password, subset):
    user = QPASEUser(pp, "alice", password)
    requests = user.toprf_request(subset)
    try:
        responses = [servers[i].toprf_evaluate(request) for i, request in requests.items()]
    except ProtocolAbort as error:
        return f"L1 refused: {error}"
    k = user.toprf_finalize(responses)
    try:
        user.login_verify(k, responses[0].user_pk, servers[subset[0]].new_session("alice"))
    except ProtocolAbort:
        return "L2: pk_u' != pk_u"
    return "logged in"


rows = []
for password in ["password1", "letmein", "qwerty123", "dragon", "monkey", "sunshine"]:
    outcome = guess(password, (1, 2))
    rows.append((password, outcome, attempts()))
    if outcome.startswith("L1 refused"):
        break
print(table(("guess", "outcome", "mu_u after"), rows))
print()
locked_out = {subset: guess(PASSWORD, subset) for subset in ((1, 2), (2, 3))}
for subset, outcome in locked_out.items():
    print(f"correct password with {subset}: {outcome}")

guess      outcome                                          mu_u after
---------  -----------------------------------------------  -------------------------------
password1  L2: pk_u' != pk_u                                {'S_1': 8, 'S_2': 14, 'S_3': 6}
letmein    L1 refused: mu_u = 14 reached the limit mu = 14  {'S_1': 9, 'S_2': 14, 'S_3': 6}



correct password with (1, 2): L1 refused: mu_u = 14 reached the limit mu = 14
correct password with (2, 3): L1 refused: mu_u = 14 reached the limit mu = 14


## Checks

In [12]:
assert login_12.k_u == login_23.k_u == registered.k_u, "Login with {1, 2} and {2, 3} gives Register's K_u"
assert login_12.user_pk == login_23.user_pk == key_message.user_pk, "and the same pk_u"
assert refused_13, "{1, 3} is refused before any server is contacted"
assert wrong_rejected and wrong_counted, "a wrong password fails L2 and still counts at L1"
assert same_k_u == same_pk == K, "every repeated login reproduces K_u and pk_u"
assert rows[-1][1].startswith("L1 refused"), "guessing stops at the limit mu"
assert all(outcome.startswith("L1 refused") for outcome in locked_out.values()), "the account is locked for the epoch"
print("all register and login checks passed")

all register and login checks passed
